# Superstore Sales Analysis and Business Insights
## Python Data Analysis & Exploratory Data Analysis (EDA)

**Objective:** Analyze sales, profit, products, regions, customers, and time trends to develop evidence-based business recommendations.

**Workflow:** Select → Understand → Clean → Analyze → Visualize → Interpret → Conclude.

**Libraries:** Pandas, NumPy, Matplotlib, Seaborn.

> Run the notebook from top to bottom. Keep the CSV file in the same folder as this notebook, or update `DATA_PATH` below.


## 1. Import libraries and load the dataset

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

pd.set_option("display.max_columns", None)
sns.set_theme(style="whitegrid")

DATA_PATH = "Sample - Superstore.csv"
df = pd.read_csv(DATA_PATH, encoding="latin1")
df.head()


## 2. Understand the dataset

In [ ]:
print("Rows and columns:", df.shape)
print("\\nColumn names:")
print(df.columns.tolist())
print("\\nData types and non-null counts:")
df.info()


In [ ]:
display(df.head())
display(df.tail())
display(df.describe(include="all").T)


**What one row represents:** One line item in an order. An order can appear on multiple rows when it contains multiple products or line items.

## 3. Initial data-quality checks

In [ ]:
print("Missing values by column:")
display(df.isna().sum().sort_values(ascending=False))
print("Exact duplicate rows:", df.duplicated().sum())
print("\\nUnique values per column:")
display(df.nunique().sort_values())


In [ ]:
# Inspect unusual values before deciding whether any action is necessary
print("Negative quantities:", (df["Quantity"] < 0).sum())
print("Negative sales:", (df["Sales"] < 0).sum())
print("Negative profit rows:", (df["Profit"] < 0).sum())
print("Discount range:", df["Discount"].min(), "to", df["Discount"].max())


## 4. Clean and prepare the data

In [ ]:
# Work on a copy so the original import remains unchanged
df_clean = df.copy()

# Parse dates; errors become NaT and are checked below
df_clean["Order Date"] = pd.to_datetime(df_clean["Order Date"], errors="coerce")
df_clean["Ship Date"] = pd.to_datetime(df_clean["Ship Date"], errors="coerce")

print("Invalid order dates:", df_clean["Order Date"].isna().sum())
print("Invalid ship dates:", df_clean["Ship Date"].isna().sum())

# Check duplicates first. Only remove exact duplicates if any exist.
duplicate_count = df_clean.duplicated().sum()
if duplicate_count:
    df_clean = df_clean.drop_duplicates().copy()
print("Exact duplicates removed:", duplicate_count)

# Useful date features
df_clean["Order Year"] = df_clean["Order Date"].dt.year
df_clean["Order Month"] = df_clean["Order Date"].dt.to_period("M").astype(str)
df_clean["Shipping Days"] = (df_clean["Ship Date"] - df_clean["Order Date"]).dt.days

# Re-check quality after preparation
display(df_clean.isna().sum().sort_values(ascending=False).head(10))
df_clean.head()


**Cleaning decision:** The supplied file has no missing values and no exact duplicate rows in the initial check. We therefore do not impute or delete records without a reason. Date columns are converted to datetime, and date-derived columns are added for analysis. Negative profit is not automatically an error; it can represent a real loss.

## 5. Statistical analysis

In [ ]:
numeric_cols = ["Sales", "Quantity", "Discount", "Profit"]
display(df_clean[numeric_cols].describe().T)

summary = pd.DataFrame({
    "Mean": df_clean[numeric_cols].mean(),
    "Median": df_clean[numeric_cols].median(),
    "Std Dev": df_clean[numeric_cols].std(),
    "Minimum": df_clean[numeric_cols].min(),
    "Maximum": df_clean[numeric_cols].max()
})
display(summary)


## 6. Univariate analysis

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4))
sns.histplot(df_clean["Sales"], bins=40, kde=True, ax=axes[0])
axes[0].set_title("Distribution of Sales")
sns.boxplot(x=df_clean["Profit"], ax=axes[1])
axes[1].set_title("Profit: Outlier and Spread Check")
plt.tight_layout()
plt.show()

plt.figure(figsize=(8, 4))
sns.countplot(data=df_clean, x="Category", order=df_clean["Category"].value_counts().index)
plt.title("Number of Line Items by Category")
plt.xticks(rotation=15)
plt.tight_layout()
plt.show()


## 7. Bivariate analysis

In [ ]:
plt.figure(figsize=(8, 5))
sns.scatterplot(data=df_clean, x="Discount", y="Profit", alpha=0.35)
plt.title("Discount vs Profit")
plt.tight_layout()
plt.show()

plt.figure(figsize=(8, 5))
sns.scatterplot(data=df_clean, x="Sales", y="Profit", hue="Category", alpha=0.6)
plt.title("Sales vs Profit by Category")
plt.tight_layout()
plt.show()


## 8. Multivariate analysis

In [ ]:
corr = df_clean[["Sales", "Quantity", "Discount", "Profit", "Shipping Days"]].corr(numeric_only=True)
plt.figure(figsize=(8, 6))
sns.heatmap(corr, annot=True, cmap="coolwarm", center=0, fmt=".2f")
plt.title("Correlation Matrix")
plt.tight_layout()
plt.show()
corr


**Interpretation note:** Correlation measures association, not cause and effect. A relationship between discount and profit alone does not prove that discounting caused a profit change.

## 9. GroupBy analysis

In [ ]:
region_summary = df_clean.groupby("Region", as_index=False).agg(
    Sales=("Sales", "sum"),
    Profit=("Profit", "sum"),
    Quantity=("Quantity", "sum"),
    Orders=("Order ID", "nunique")
).sort_values("Sales", ascending=False)
display(region_summary)

category_summary = df_clean.groupby("Category", as_index=False).agg(
    Sales=("Sales", "sum"),
    Profit=("Profit", "sum"),
    Avg_Sales=("Sales", "mean")
).sort_values("Profit", ascending=False)
display(category_summary)

segment_summary = df_clean.groupby("Segment", as_index=False).agg(
    Sales=("Sales", "sum"), Profit=("Profit", "sum")
).sort_values("Sales", ascending=False)
display(segment_summary)


## 10. Sorting and ranking

In [ ]:
print("Top 5 products by total sales")
display(df_clean.groupby("Product Name", as_index=False).agg(
    Sales=("Sales", "sum"), Profit=("Profit", "sum")
).sort_values("Sales", ascending=False).head(5))

print("Bottom 5 sub-categories by total profit")
display(df_clean.groupby("Sub-Category", as_index=False).agg(
    Sales=("Sales", "sum"), Profit=("Profit", "sum")
).sort_values("Profit").head(5))


## 11. Pivot table and crosstab

In [ ]:
sales_pivot = pd.pivot_table(
    df_clean, values="Sales", index="Region", columns="Category",
    aggfunc="sum", fill_value=0
)
display(sales_pivot)

# Count of line items by customer segment and ship mode
display(pd.crosstab(df_clean["Segment"], df_clean["Ship Mode"]))


## 12. Time-series analysis

In [ ]:
monthly = df_clean.groupby("Order Month", as_index=False).agg(
    Sales=("Sales", "sum"), Profit=("Profit", "sum")
)
plt.figure(figsize=(13, 5))
plt.plot(monthly["Order Month"], monthly["Sales"], marker="o", label="Sales")
plt.title("Monthly Sales Trend")
plt.xlabel("Order Month")
plt.ylabel("Sales")
plt.xticks(range(0, len(monthly), 4), monthly["Order Month"].iloc[::4], rotation=45)
plt.legend()
plt.tight_layout()
plt.show()

display(monthly.sort_values("Sales", ascending=False).head(5))


## 13. NumPy operations

In [ ]:
sales_array = df_clean["Sales"].to_numpy()
print("Total sales:", np.sum(sales_array))
print("Mean sales per line item:", np.mean(sales_array))
print("Median sales per line item:", np.median(sales_array))
print("Minimum line-item sales:", np.min(sales_array))
print("Maximum line-item sales:", np.max(sales_array))
print("Sales standard deviation:", np.std(sales_array))

# Boolean masking: count of high-sales line items
print("Line items with sales above $500:", np.sum(sales_array > 500))


## 14. Key findings

In [ ]:
total_sales = df_clean["Sales"].sum()
total_profit = df_clean["Profit"].sum()
total_orders = df_clean["Order ID"].nunique()
total_customers = df_clean["Customer ID"].nunique()
profit_margin = total_profit / total_sales * 100

print(f"Total sales: ${total_sales:,.2f}")
print(f"Total profit: ${total_profit:,.2f}")
print(f"Profit margin: {profit_margin:.2f}%")
print(f"Unique orders: {total_orders:,}")
print(f"Unique customers: {total_customers:,}")

display(region_summary)
display(category_summary)
display(monthly.sort_values("Sales", ascending=False).head(5))


## 15. Business recommendations

Use the computed tables and charts to support recommendations. Investigate low-profit sub-categories before changing pricing or promotions. Compare discount levels with profit, but account for product mix and order size. Review monthly trends for seasonal planning. Prioritize regions and customer segments using both sales and profit rather than sales alone.

## 16. Conclusion

This project demonstrates an end-to-end data-analysis workflow: loading and understanding data, checking quality, preparing date fields, descriptive statistics, univariate/bivariate/multivariate analysis, GroupBy, sorting, pivot tables, crosstabs, NumPy calculations, visualizations, correlation, and business interpretation.

**Portfolio reminder:** Only claim the work you actually run, verify, and understand. Include selected charts, the notebook, a short README, and your own interpretation when publishing.
